# D* Concept Demo: Replanning After Map Changes

This notebook is a guided learning walkthrough for the dynamic-replanning problem that D* was designed to solve.

Learning goals:
- observe how map updates invalidate an existing route,
- compare search effort before and after an update,
- understand why incremental methods (D*, D* Lite) matter.

Note: the implementation uses repeated A* as a baseline for clarity.

In [ ]:
%matplotlib inline

import heapq
import math

import matplotlib.pyplot as plt
import numpy as np

## 1. Build a grid map

We start with a map that has obstacles and narrow passages so replanning effects are visible.

In [ ]:
H, W = 35, 35
grid = np.zeros((H, W), dtype=np.uint8)

# Static obstacles
grid[8:26, 10] = 1
grid[8:26, 24] = 1
grid[18, 10:25] = 1

# Create doorways so the map remains solvable
grid[12, 10] = 0
grid[22, 24] = 0
grid[18, 16] = 0

start = (3, 3)
goal = (31, 31)

start, goal

## 2. Baseline planner (A*)

This helper code provides the baseline search we will rerun before and after map changes.

Focus point:
- In true D*, most of this work is updated incrementally rather than recomputed from scratch.

In [ ]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c, h, w):
    return 0 <= r < h and 0 <= c < w

def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def reconstruct(parent, node):
    path = [node]
    while node in parent:
        node = parent[node]
        path.append(node)
    path.reverse()
    return path

def astar(grid_map, start_node, goal_node):
    h, w = grid_map.shape
    open_heap = []
    parent = {}
    g = {start_node: 0.0}
    closed = set()

    heapq.heappush(open_heap, (manhattan(start_node, goal_node), 0.0, start_node))

    expansions = 0

    while open_heap:
        _, curr_g, current = heapq.heappop(open_heap)
        if current in closed:
            continue

        closed.add(current)
        expansions += 1

        if current == goal_node:
            path = reconstruct(parent, current)
            return path, g[current], expansions

        for dr, dc in DIRS:
            nr, nc = current[0] + dr, current[1] + dc
            if not in_bounds(nr, nc, h, w):
                continue
            if grid_map[nr, nc] == 1:
                continue

            nxt = (nr, nc)
            new_g = curr_g + 1.0

            if new_g < g.get(nxt, math.inf):
                g[nxt] = new_g
                parent[nxt] = current
                f = new_g + manhattan(nxt, goal_node)
                heapq.heappush(open_heap, (f, new_g, nxt))

    return None, math.inf, expansions

## 3. Plot utility

This helper keeps visual comparisons consistent between the original and updated map.

In [ ]:
def draw_scene(ax, grid_map, path=None, title=''):
    ax.imshow(grid_map, cmap='Greys', origin='upper')
    ax.scatter(start[1], start[0], c='limegreen', s=80, marker='o', label='start')
    ax.scatter(goal[1], goal[0], c='crimson', s=80, marker='*', label='goal')

    if path is not None and len(path) > 0:
        cols = [p[1] for p in path]
        rows = [p[0] for p in path]
        ax.plot(cols, rows, color='deepskyblue', linewidth=2.0, label='path')

    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(loc='upper right')

## 4. Initial planning run

Run the baseline planner once and note the initial path cost and node expansions.

In [ ]:
path_before, cost_before, exp_before = astar(grid, start, goal)

print(f'Initial path found: {path_before is not None}')
print(f'Initial path cost: {cost_before:.1f}')
print(f'Initial node expansions: {exp_before}')

## 5. Simulate map update

We now inject new obstacles on part of the previous route to mimic online map discovery.

Reflection prompt:
- Which part of the old path becomes invalid, and why?

In [ ]:
grid_updated = grid.copy()

# Block a middle segment of the previous path while keeping start/goal free.
if path_before is not None and len(path_before) > 10:
    segment = path_before[len(path_before)//3 : len(path_before)//3 + 8]
    for r, c in segment:
        if (r, c) != start and (r, c) != goal:
            grid_updated[r, c] = 1

## 6. Replan on the updated map

Run the same planner again after the update and compare feasibility, cost, and expansions.

In [ ]:
path_after, cost_after, exp_after = astar(grid_updated, start, goal)

print(f'Replanned path found: {path_after is not None}')
print(f'Replanned path cost: {cost_after:.1f}')
print(f'Replanned node expansions: {exp_after}')

## 7. Visual comparison

Inspect both maps side by side to see how the route changed after new obstacles appeared.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
draw_scene(axes[0], grid, path_before, title='Before update')
draw_scene(axes[1], grid_updated, path_after, title='After update and replanning')
plt.tight_layout()
plt.show()

## 8. Discussion and self-check

What this notebook showed:
- Dynamic updates can invalidate a previously good route.
- Rerunning A* works but may repeat a lot of search effort.
- D* methods aim to repair search state incrementally.

Try it yourself:
- Increase the blocked segment length and rerun.
- Move the update to a different part of the map and compare expansion counts.